# Build the immutable Internvl 8B snapshot

Settings: **Accelerator OFF**, **Internet ON**. Attach the authenticated CertVIC CODE bundle under any account, title, filename, extension, mount, or nesting. Run All without editing. Canonical output: `internvl2_8b_snapshot.zip`.


In [ ]:
import hashlib
import json
import os
import pathlib
import platform
import shutil
import stat
import sys
import zipfile

print(json.dumps({
    "status": "IMMEDIATE_SNAPSHOT_PROVISIONING_PROBE",
    "executable": sys.executable,
    "implementation": platform.python_implementation(),
    "python": platform.python_version(),
    "architecture": platform.machine(),
    "system": platform.system(),
    "libc": platform.libc_ver(),
}, indent=2))
if platform.python_implementation() != "CPython" or not platform.python_version().startswith("3.12."):
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: CPython 3.12 required")
if platform.system() != "Linux" or platform.machine().lower() != "x86_64":
    raise RuntimeError("CERTVIC_RUNTIME_01_PYTHON_PROFILE_NOT_SUPPORTED: Linux x86_64 required")

ROOTS = [
    pathlib.Path(value)
    for value in os.environ.get("CERTVIC_INPUT_ROOTS", "/kaggle/input").split(os.pathsep)
    if pathlib.Path(value).is_dir()
]
if not ROOTS:
    raise RuntimeError("CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND: no input roots")

def _safe_name(info):
    name = info.filename
    pure = pathlib.PurePosixPath(name)
    mode = (info.external_attr >> 16) & 0xFFFF
    if (not name or name.endswith("/") or pure.is_absolute() or ".." in pure.parts
            or info.is_dir() or stat.S_ISLNK(mode)):
        raise RuntimeError(f"CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: unsafe {name!r}")
    return pure.as_posix()

def _authenticate_code(candidate):
    try:
        with zipfile.ZipFile(candidate) as archive:
            infos = archive.infolist()
            names = [_safe_name(info) for info in infos]
            if len(names) != len(set(names)) or archive.testzip() is not None:
                return None
            if "bundle_manifest.json" not in names or "hash_manifest.json" not in names:
                return None
            manifest = json.loads(archive.read("bundle_manifest.json"))
            hashes = json.loads(archive.read("hash_manifest.json"))
            if (manifest.get("schema") != "certvic.kaggle.bundle.v1"
                    or manifest.get("bundle_type") != "CODE"
                    or hashes.get("schema") != "certvic.kaggle.hash_manifest.v1"):
                return None
            declared = hashes.get("files", {})
            if set(names) != set(declared) | {"hash_manifest.json"}:
                raise RuntimeError("CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: file universe")
            for name, record in declared.items():
                payload = archive.read(name)
                if record != {"size": len(payload), "sha256": hashlib.sha256(payload).hexdigest()}:
                    raise RuntimeError(f"CERTVIC_DISCOVERY_03_CONTENT_AUTHENTICATION_FAILED: {name}")
            return hashlib.sha256(candidate.read_bytes()).hexdigest()
    except zipfile.BadZipFile:
        return None

matches = []
for root in ROOTS:
    for candidate in root.rglob("*"):
        if candidate.is_file() and not candidate.is_symlink():
            identity = _authenticate_code(candidate)
            if identity:
                matches.append((candidate.resolve(), identity))
identities = sorted({identity for _, identity in matches})
if not identities:
    raise RuntimeError("CERTVIC_DISCOVERY_01_REQUIRED_ROLE_NOT_FOUND: CODE")
if len(identities) != 1:
    raise RuntimeError("CERTVIC_DISCOVERY_02_AMBIGUOUS_DISTINCT_CONTENT: CODE")
selected = next(path for path, identity in matches if identity == identities[0])
PROJECT_ROOT = pathlib.Path("/kaggle/working/certvic_snapshot_provisioning_code")
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(selected) as archive:
    for info in archive.infolist():
        name = _safe_name(info)
        destination = (PROJECT_ROOT / name).resolve()
        destination.relative_to(PROJECT_ROOT.resolve())
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(info) as reader, destination.open("xb") as writer:
            shutil.copyfileobj(reader, writer)
sys.path.insert(0, str(PROJECT_ROOT))
print({
    "status": "CONTENT_AUTHENTICATED_ANY_LOCATION",
    "role": "CODE",
    "content_identity_sha256": identities[0],
    "observed_dataset_folder": str(selected.parent),
    "observed_archive_name": selected.name,
    "mirrors": len(matches),
})


In [ ]:
import subprocess
from pathlib import Path

PROVIDER = 'internvl_8b'
MODEL_REPOSITORY = 'OpenGVLab/InternVL2-8B'
MODEL_COMMIT = '6fb9ad6924f69424e57fab2ab061d707688f0296'
PROCESSOR_COMMIT = MODEL_COMMIT
CANONICAL_OUTPUT = 'internvl2_8b_snapshot.zip'

subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub==0.26.2"
], check=True)
from huggingface_hub import snapshot_download
from certvic.cvpr.kaggle_bundle import verify_bundle
from certvic.cvpr.snapshot_bundle_builder import build_snapshot_bundle

snapshot_root = Path("/kaggle/working/model_snapshot") / PROVIDER
snapshot_download(
    repo_id=MODEL_REPOSITORY,
    revision=MODEL_COMMIT,
    local_dir=snapshot_root,
)
symlinks = [str(path) for path in snapshot_root.rglob("*") if path.is_symlink()]
if symlinks:
    raise RuntimeError(f"downloaded snapshot contains symlinks: {symlinks[:5]}")
output = Path("/kaggle/working") / CANONICAL_OUTPUT
rebuild = output.with_name(output.stem + ".deterministic_rebuild.zip")
first = build_snapshot_bundle(
    PROVIDER,
    snapshot_root,
    model_commit=MODEL_COMMIT,
    processor_commit=PROCESSOR_COMMIT,
    output=output,
)
second = build_snapshot_bundle(
    PROVIDER,
    snapshot_root,
    model_commit=MODEL_COMMIT,
    processor_commit=PROCESSOR_COMMIT,
    output=rebuild,
)
if output.read_bytes() != rebuild.read_bytes():
    raise RuntimeError("snapshot deterministic rebuild is not byte-identical")
rebuild.unlink()
verification = verify_bundle(output)
if not verification["passed"]:
    raise RuntimeError(f"snapshot bundle verification failed: {verification['errors']}")
print(json.dumps({
    "status": "IMMUTABLE_SNAPSHOT_BUILT_DETERMINISTIC",
    "provider": PROVIDER,
    "model_repository": MODEL_REPOSITORY,
    "model_commit": MODEL_COMMIT,
    "processor_commit": PROCESSOR_COMMIT,
    "canonical_output": str(output),
    "size": output.stat().st_size,
    "sha256": first["sha256"],
    "deterministic_rebuild": True,
    "paper_evidence": False,
}, indent=2, sort_keys=True))
print("NEXT: download " + CANONICAL_OUTPUT + ", then run the matching provider-specific 00B notebook with Accelerator OFF and Internet OFF.")
